In [1]:
import os
import sys
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

dir = "/Users/julianguyen/Documents/BCaATAC"
os.chdir(dir)
sys.path.insert(0, dir)
from utils.gene_surrogate.helpers import *

np.random.seed(101)

### Load in and format matrices

In [2]:
# load in meta
meta = pd.read_csv("data/rawdata/tcga/TCGA_sourcefiles.csv")

# load in ARCHE scores
arche_scores = pd.read_csv("data/procdata/NMF/ATAC_heatmap_rank6.png.order.matrix", sep='\t')
arche_scores.index = [f"ARCHE{i}" for i in range(1, 7)]

# match sample IDs
lookup = dict(zip(meta['ATAC.Seq.File.Name'], meta['Sample.Name']))
arche_scores.columns = [lookup.get(col.lstrip('X'), None) for col in arche_scores.columns]

# remove duplicate ATAC-seq
dupe = arche_scores.columns[arche_scores.columns.duplicated()]
arche_scores = arche_scores.drop(columns=dupe)

# load in rna and match samples
full_cohort = pd.read_csv("data/rawdata/tcga/Human__TCGA_BRCA__UNC__RNAseq__HiSeq_RNA__01_28_2016__BI__Gene__Firehose_RSEM_log2.cct", sep='\t', index_col=0)
rna = full_cohort.loc[:, full_cohort.columns.isin(arche_scores.columns)]
full_cohort = full_cohort.drop(columns=arche_scores.columns)

# transpose matrices
full_cohort = full_cohort.T
arche_scores = arche_scores.T
rna = rna.T
rna = rna.reindex(arche_scores.index)

In [3]:
arche_scores.head()

,ARCHE1,ARCHE2,ARCHE3,ARCHE4,ARCHE5,ARCHE6
TCGA.A2.A0YT,5.32,0.00,0.0,0.0,0.0,0.0
TCGA.AO.A0J5,5.16,0.02,0.0,0.0,0.0,0.0
TCGA.A2.A0YK,4.90,0.71,0.0,0.0,0.0,0.0
TCGA.BH.A0BA,4.81,0.00,0.0,0.0,0.0,0.0
TCGA.A2.A0ES,4.64,0.00,0.0,0.0,0.0,0.0


In [4]:
rna.head()

attrib_name,A1BG,A1CF,A2BP1,A2LD1,A2ML1,A2M,A4GALT,A4GNT,AAA1,AAAS,...,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,psiTPTE22,tAKR
TCGA.A2.A0YT,7.2699,0.0000,0.0000,4.9454,5.9020,13.8519,7.9947,0.0000,0.0,9.2469,...,6.4119,9.1288,10.4603,7.6324,10.3355,10.9964,10.6943,9.8862,8.5727,0.0
TCGA.AO.A0J5,7.1214,0.0000,0.6422,7.2256,0.6422,13.8540,8.2186,1.0850,0.0,9.3236,...,6.4762,9.0430,10.2440,4.2435,10.1240,11.7641,10.5799,10.1186,6.4347,0.0
TCGA.A2.A0YK,6.8112,0.0000,0.0000,6.1013,4.8487,14.8459,8.0462,1.0591,0.0,9.0084,...,6.6011,9.1362,10.4478,5.6885,10.1256,12.0782,10.9735,10.1711,7.7616,0.0
TCGA.BH.A0BA,6.9718,0.4251,0.0000,6.1035,3.8786,13.7898,7.3915,0.7532,0.0,9.4480,...,6.6146,9.3507,10.1381,6.6146,9.8549,11.4044,10.4523,9.7916,7.0987,0.0
TCGA.A2.A0ES,7.0390,0.0000,0.4558,6.4260,2.2373,15.0711,8.7183,1.5148,0.0,9.2542,...,6.4399,9.3227,10.3694,5.8165,10.0480,12.2864,10.5378,9.9991,7.5042,0.0


### Featre selection

In [5]:
arche1 = important_features(rna, arche_scores["ARCHE1"], thres = 0.35)
arche1 = corr_features(arche1, 0.8)

Selected threshold: 0.35
Num features with positive correlation > threshold: 215
Num features with negative correlation > threshold: 70
Total number of features: 285

Num correlated features: 50
Original number of feaures: 285
Number of features remaining: 235


In [6]:
arche2 = important_features(rna, arche_scores["ARCHE2"], thres = 0.6)
arche2 = corr_features(arche2, 0.8)

Selected threshold: 0.6
Num features with positive correlation > threshold: 185
Num features with negative correlation > threshold: 137
Total number of features: 322

Num correlated features: 48
Original number of feaures: 322
Number of features remaining: 274


In [7]:
arche3 = important_features(rna, arche_scores["ARCHE3"], thres = 0.4)
arche3 = corr_features(arche3, 0.8)

Selected threshold: 0.4
Num features with positive correlation > threshold: 199
Num features with negative correlation > threshold: 69
Total number of features: 268

Num correlated features: 27
Original number of feaures: 268
Number of features remaining: 241


In [8]:
arche4 = important_features(rna, arche_scores["ARCHE4"], thres = 0.4)
arche4 = corr_features(arche4, 0.8)

Selected threshold: 0.4
Num features with positive correlation > threshold: 163
Num features with negative correlation > threshold: 156
Total number of features: 319

Num correlated features: 25
Original number of feaures: 319
Number of features remaining: 294


In [9]:
arche5 = important_features(rna, arche_scores["ARCHE5"], thres = 0.4)
arche5 = corr_features(arche5, 0.8)

Selected threshold: 0.4
Num features with positive correlation > threshold: 512
Num features with negative correlation > threshold: 72
Total number of features: 584

Num correlated features: 250
Original number of feaures: 584
Number of features remaining: 334


In [10]:
arche6 = important_features(rna, arche_scores["ARCHE6"], thres = 0.45)
arche6 = corr_features(arche6, 0.8)

Selected threshold: 0.45
Num features with positive correlation > threshold: 198
Num features with negative correlation > threshold: 54
Total number of features: 252

Num correlated features: 24
Original number of feaures: 252
Number of features remaining: 228


### Elastic Net Models

In [11]:
run_elastic_net(arche1, arche_scores["ARCHE1"], "1-Signatures/geneSurrogates/arche1_", full_cohort)
run_elastic_net(arche2, arche_scores["ARCHE2"], "1-Signatures/geneSurrogates/arche2_", full_cohort)
run_elastic_net(arche3, arche_scores["ARCHE3"], "1-Signatures/geneSurrogates/arche3_", full_cohort)
run_elastic_net(arche4, arche_scores["ARCHE4"], "1-Signatures/geneSurrogates/arche4_", full_cohort)
run_elastic_net(arche5, arche_scores["ARCHE5"], "1-Signatures/geneSurrogates/arche5_", full_cohort)
run_elastic_net(arche6, arche_scores["ARCHE6"], "1-Signatures/geneSurrogates/arche6_", full_cohort)